# Reasoning Evaluation - Nepali

In [1]:
import os
import sys
import math
import json
import string
import argparse
import logging
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
import sentencepiece as spm
from typing import Dict, List, Any

# ==========================================
# DEFAULT PATHS (Fallback / Kaggle)
# ==========================================
PRETRAINED_CHECKPOINT = "/kaggle/input/notebooks/srinivasu2005/train-nepali/checkpoints_nepali/best_model.pt"
FINETUNED_CHECKPOINT = "/kaggle/input/notebooks/srinivasu2005/finetune-nepali-path/finetuned_model_nepali_7k/best_finetuned_model.pt"
TEST_JSONL = "/kaggle/input/datasets/srinivasu2005/finetune-nepali/test.jsonl"
TOKENIZER_PATH = "/kaggle/input/notebooks/srinivasu2005/nepali-tokenizer/tokenizer/tokenizer.model"
OUTPUT_DIR = "eval_nepali_finetuned"

## Model Architecture

In [2]:
VOCAB_SIZE = 4000
D_FF = 1184

@dataclass
class TransformerConfig:
    vocab_size: int = VOCAB_SIZE; d_model: int = 512; n_layers: int = 8
    n_heads: int = 8; d_ff: int = D_FF; max_seq_len: int = 512
    dropout: float = 0.0; tie_weights: bool = True; rope_base: int = 10000
    pad_id: int = 0; bos_id: int = 2; eos_id: int = 3
    @property
    def d_k(self): return self.d_model // self.n_heads

def precompute_rope_frequencies(d_k, max_seq_len, base=10000, device=None):
    freq = torch.arange(0, d_k, 2, dtype=torch.float32, device=device)
    theta = 1.0 / (base ** (freq / d_k))
    pos = torch.arange(0, max_seq_len, dtype=torch.float32, device=device)
    angles = torch.outer(pos, theta)
    return torch.cos(angles), torch.sin(angles)

def apply_rope(x, cos, sin):
    x1, x2 = x[..., 0::2], x[..., 1::2]
    c = cos.unsqueeze(0).unsqueeze(0); s = sin.unsqueeze(0).unsqueeze(0)
    return torch.stack([x1*c - x2*s, x1*s + x2*c], dim=-1).flatten(-2)

class MultiHeadCausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_heads, self.d_k, self.d_model = cfg.n_heads, cfg.d_k, cfg.d_model
        self.W_Q = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.W_K = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.W_V = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.W_O = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.attn_dropout = nn.Dropout(cfg.dropout)
        self.resid_dropout = nn.Dropout(cfg.dropout)
        cos, sin = precompute_rope_frequencies(cfg.d_k, cfg.max_seq_len, cfg.rope_base)
        self.register_buffer("cos_table", cos); self.register_buffer("sin_table", sin)
        self.register_buffer("causal_mask", torch.triu(torch.ones(cfg.max_seq_len, cfg.max_seq_len), diagonal=1).bool())
    def forward(self, x, return_attention=False):
        B, T, _ = x.shape
        Q = self.W_Q(x).view(B,T,self.n_heads,self.d_k).transpose(1,2)
        K = self.W_K(x).view(B,T,self.n_heads,self.d_k).transpose(1,2)
        V = self.W_V(x).view(B,T,self.n_heads,self.d_k).transpose(1,2)
        c, s = self.cos_table[:T], self.sin_table[:T]
        Q, K = apply_rope(Q,c,s), apply_rope(K,c,s)
        sc = torch.matmul(Q, K.transpose(-2,-1)) / math.sqrt(self.d_k)
        sc = sc.masked_fill(self.causal_mask[:T,:T].unsqueeze(0).unsqueeze(0), float('-inf'))
        aw = torch.softmax(sc, dim=-1)
        ctx = torch.matmul(self.attn_dropout(aw), V).transpose(1,2).contiguous().view(B,T,self.d_model)
        return (self.resid_dropout(self.W_O(ctx)), aw) if return_attention else (self.resid_dropout(self.W_O(ctx)), None)

class SwiGLUFeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.W_gate = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.W_up = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.W_down = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)
        self.dropout = nn.Dropout(cfg.dropout)
    def forward(self, x): return self.dropout(self.W_down(F.silu(self.W_gate(x)) * self.W_up(x)))

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model); self.attn = MultiHeadCausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model); self.ffn = SwiGLUFeedForward(cfg)
    def forward(self, x, return_attention=False):
        ao, aw = self.attn(self.ln1(x), return_attention); x = x + ao; x = x + self.ffn(self.ln2(x)); return x, aw

class DecoderOnlyTransformer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.config = cfg
        self.embedding = nn.Embedding(cfg.vocab_size, cfg.d_model, padding_idx=cfg.pad_id)
        self.emb_dropout = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layers)])
        self.final_ln = nn.LayerNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_weights: self.lm_head.weight = self.embedding.weight
    def forward(self, input_ids, targets=None, return_attention=False):
        x = self.emb_dropout(self.embedding(input_ids))
        all_attn = [] if return_attention else None
        for block in self.blocks:
            x, aw = block(x, return_attention)
            if return_attention and aw is not None: all_attn.append(aw)
        logits = self.lm_head(self.final_ln(x))
        result = {"logits": logits}
        if targets is not None:
            result["loss"] = F.cross_entropy(logits.view(-1, self.config.vocab_size), targets.view(-1), ignore_index=self.config.pad_id)
        if return_attention: result["attention_weights"] = all_attn
        return result
    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens=128, temperature=0.0, top_k=0, repetition_penalty=1.0):
        self.eval(); gen = input_ids
        for _ in range(max_new_tokens):
            ctx = gen if gen.shape[1] <= self.config.max_seq_len else gen[:, -self.config.max_seq_len:]
            logits = self.forward(ctx)["logits"][:, -1, :]
            if repetition_penalty != 1.0:
                for b in range(input_ids.size(0)):
                    for token_id in set(gen[b].tolist()):
                        if logits[b, token_id] < 0: logits[b, token_id] *= repetition_penalty
                        else: logits[b, token_id] /= repetition_penalty
            if temperature == 0.0: nt = logits.argmax(dim=-1, keepdim=True)
            else:
                logits = logits / temperature
                if top_k > 0:
                    v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                    logits[logits < v[:, [-1]]] = float('-inf')
                nt = torch.multinomial(torch.softmax(logits, dim=-1), 1)
            gen = torch.cat([gen, nt], dim=1)
            if (nt == self.config.eos_id).all(): break
        return gen

## Evaluation Logic

In [3]:
def load_model(checkpoint_path, device="cpu"):
    cfg = TransformerConfig(dropout=0.0)
    model = DecoderOnlyTransformer(cfg).to(device)
    if os.path.exists(checkpoint_path):
        try:
            state_dict = torch.load(checkpoint_path, map_location=device, weights_only=False)
        except TypeError:
            state_dict = torch.load(checkpoint_path, map_location=device)
        if isinstance(state_dict, dict) and "model_state_dict" in state_dict:
            state_dict = state_dict["model_state_dict"]
        model.load_state_dict(state_dict)
    else:
        print(f"Warning: Checkpoint not found at '{checkpoint_path}'. Evaluating randomly initialized model.")
    model.eval()
    return model

def normalize_text(text):
    return text.strip().strip(string.punctuation)

def extract_answer(generated_text, separator="उत्तर:"):
    if separator in generated_text:
        return generated_text.split(separator)[-1].strip()
    return generated_text.strip()

def categorize_failure(pred, gold, question):
    pred_norm, gold_norm = normalize_text(pred), normalize_text(gold)
    if not pred_norm: return "incomplete"
    q_words = set(question.split())
    if any(w in q_words for w in pred_norm.split()) and pred_norm != gold_norm:
        return "wrong_entity"
    opposites = [("हो", "होइन"), ("दायाँ", "बायाँ"), ("माथि", "तल"), ("सत्य", "असत्य")]
    for w1, w2 in opposites:
        if (w1 in gold_norm and w2 in pred_norm) or (w2 in gold_norm and w1 in pred_norm):
            return "wrong_direction"
    if len(pred_norm) > len(gold_norm) * 3 or len(pred_norm) > 100:
        return "hallucination"
    return "other"

def run_evaluation(args=None):
    pretrained_checkpoint = getattr(args, "pretrained_checkpoint", None) or PRETRAINED_CHECKPOINT
    finetuned_checkpoint = getattr(args, "finetuned_checkpoint", None) or FINETUNED_CHECKPOINT
    test_jsonl = getattr(args, "test_data", None) or getattr(args, "test_jsonl", None) or TEST_JSONL
    tokenizer_path = getattr(args, "tokenizer_path", None) or getattr(args, "tokenizer", None) or TOKENIZER_PATH
    output_dir = getattr(args, "output_dir", None) or OUTPUT_DIR
    max_samples = getattr(args, "max_samples", None)
    device_name = getattr(args, "device", None)

    if device_name:
        device = torch.device(device_name)
    else:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    os.makedirs(output_dir, exist_ok=True)

    if not os.path.exists(tokenizer_path):
        print(f"Error: Tokenizer not found at: {tokenizer_path}")
        return

    sp = spm.SentencePieceProcessor()
    sp.load(tokenizer_path)
    print(f"Loaded tokenizer from: {tokenizer_path}")

    models = {
        "pretrained": load_model(pretrained_checkpoint, device),
        "finetuned": load_model(finetuned_checkpoint, device)
    }

    examples = []
    if os.path.exists(test_jsonl):
        with open(test_jsonl, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    examples.append(json.loads(line))
        print(f"Loaded {len(examples)} test examples from: {test_jsonl}")
    else:
        print(f"Warning: {test_jsonl} not found. Running with empty list.")

    if max_samples and max_samples > 0:
        examples = examples[:max_samples]
        print(f"Limiting evaluation to first {len(examples)} examples.")

    results = {}

    for model_name, model in models.items():
        correct, total = 0, 0
        task_correct = {}
        task_total = {}
        success_examples = []
        failure_examples = []
        failure_counts = {"wrong_entity": 0, "wrong_direction": 0, "hallucination": 0, "incomplete": 0, "other": 0}

        for ex in examples:
            inp = ex.get("input", ex.get("question", "")).strip()
            if not inp.endswith("उत्तर:"):
                q = f"{inp} उत्तर: "
            else:
                q = f"{inp} "
            gold = ex.get("target", ex.get("answer", "")).strip()
            task_type = ex.get("task_type", "unknown")

            bos = sp.bos_id() if sp.bos_id() != -1 else 2
            input_ids = [bos] + sp.encode(q)
            input_tensor = torch.tensor([input_ids], dtype=torch.long, device=device)

            out_tensor = model.generate(input_tensor, max_new_tokens=64, temperature=0.0)

            gen_ids = out_tensor[0][len(input_ids):].tolist()
            eos = sp.eos_id() if sp.eos_id() != -1 else 3
            if eos in gen_ids:
                gen_ids = gen_ids[:gen_ids.index(eos)]
            pred = sp.decode(gen_ids).strip()
            if not pred:
                full_out = sp.decode(out_tensor[0].tolist())
                pred = extract_answer(full_out, "उत्तर:")

            is_correct = normalize_text(pred) == normalize_text(gold)

            total += 1
            task_total[task_type] = task_total.get(task_type, 0) + 1
            if is_correct:
                correct += 1
                task_correct[task_type] = task_correct.get(task_type, 0) + 1
                if len(success_examples) < 10:
                    success_examples.append({"question": q, "gold": gold, "pred": pred, "task_type": task_type})
            else:
                cat = categorize_failure(pred, gold, q)
                failure_counts[cat] += 1
                if len(failure_examples) < 10:
                    failure_examples.append({"question": q, "gold": gold, "pred": pred, "category": cat, "task_type": task_type})

        acc = correct / max(total, 1)
        per_task = {k: (task_correct.get(k, 0) / task_total[k]) if task_total[k] > 0 else 0.0 for k in task_total}

        results[model_name] = {
            "accuracy": acc,
            "total_samples": total,
            "correct_samples": correct,
            "per_task_type": per_task,
            "success_examples": success_examples,
            "failure_examples": failure_examples,
            "failure_counts": failure_counts
        }
        print(f"\n{'='*20} {model_name.upper()} {'='*20}")
        print(f"Overall Accuracy: {acc:.2%} ({correct}/{total})")
        print("Task-wise Breakdown:")
        for k, v in per_task.items():
            print(f"  - {k}: {v:.2%} ({task_correct.get(k, 0)}/{task_total[k]})")
        print(f"Failure Breakdown: {failure_counts}")

    results_path = os.path.join(output_dir, "eval_results.json")
    with open(results_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"\nEvaluation complete. Results saved to: {results_path}")

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Evaluate Pretrained vs Finetuned Nepali Model on Reasoning")
    parser.add_argument("--pretrained_checkpoint", type=str, default=PRETRAINED_CHECKPOINT, help="Path to pretrained model checkpoint")
    parser.add_argument("--finetuned_checkpoint", type=str, default=FINETUNED_CHECKPOINT, help="Path to finetuned model checkpoint")
    parser.add_argument("--test_data", "--test_jsonl", dest="test_data", type=str, default=TEST_JSONL, help="Path to test.jsonl")
    parser.add_argument("--tokenizer_path", "--tokenizer", dest="tokenizer_path", type=str, default=TOKENIZER_PATH, help="Path to tokenizer.model")
    parser.add_argument("--output_dir", type=str, default=OUTPUT_DIR, help="Directory to save evaluation results")
    parser.add_argument("--max_samples", type=int, default=None, help="Maximum number of test samples to evaluate")
    parser.add_argument("--device", type=str, default=None, help="Device to run eval on ('cuda' or 'cpu')")
 
    args, _ = parser.parse_known_args()

    run_evaluation(args)

Loaded tokenizer from: /kaggle/input/notebooks/srinivasu2005/nepali-tokenizer/tokenizer/tokenizer.model
Loaded 1500 test examples from: /kaggle/input/datasets/srinivasu2005/finetune-nepali/test.jsonl

==================== PRETRAINED ====================
Overall Accuracy: 0.00% (0/1500)
Task-wise Breakdown:
  - direct_comparison: 0.00% (0/247)
  - negated: 0.00% (0/232)
  - multi_attribute: 0.00% (0/256)
  - multihop_chained: 0.00% (0/257)
  - transitive_ordering: 0.00% (0/247)
  - superlative: 0.00% (0/261)
Failure Breakdown: {'wrong_entity': 553, 'wrong_direction': 0, 'hallucination': 107, 'incomplete': 708, 'other': 132}

==================== FINETUNED ====================
Overall Accuracy: 54.33% (815/1500)
Task-wise Breakdown:
  - direct_comparison: 34.82% (86/247)
  - negated: 77.59% (180/232)
  - multi_attribute: 27.73% (71/256)
  - multihop_chained: 85.60% (220/257)
  - transitive_ordering: 78.14% (193/247)
  - superlative: 24.90% (65/261)
Failure Breakdown: {'wrong_entity': 199

In [4]:
%tb

No traceback available to show.
